# EVE 310 - Lab 05: Linear regression

**Module 2 | 09/24/2026**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThyanRevolter/eve310-fall-2026/blob/main/labs/lab05-linear-regression/notebooks/lab05-tutorial.ipynb)

## Learning objectives

By the end of this lab you will be able to:

1. Fit a univariate linear model with scikit-learn and a train/test split
2. Compute R² and adjusted R² on training and test sets
3. Encode a categorical predictor with one-hot encoding and compare models

## Before you start

1. Click **Copy to Drive** at the top of this window and work in the copy that opens. Colab throws away anything you did not copy when the runtime ends.
2. Run the setup cell below before anything else. It creates `DATA_DIR` and `FIGURES_DIR` for this lab.
3. Work down the notebook in order and run every cell. There is nothing to fill in.
4. Nothing to submit for this notebook. The **activity** notebook is the one you download as `.ipynb` and upload to Gradescope (optional, for feedback).


This lab builds the model from lecture in Python: the univariate model h(x) = β₀ + β₁x, fitted by least squares on a 70% / 30% train/test split. `sklearn` does the fitting. New today: R² and adjusted R², plots of predicted vs measured values and of residuals, and a model with a categorical variable (one-hot encoding).

The data are a traffic example that you will see again in Lecture 10: people per household (x) and average daily trips (y) for 15 households. The `nwork` vector (none / part-time / full-time) is constructed for those same 15 observations so the one-hot-encoding steps run; the original category table is in the lab slides.


## 0. Setup


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# EVE 310 setup - run this cell first, every time you open this notebook.
import pathlib

LAB = "lab05-linear-regression"

DATA_DIR = pathlib.Path("data")
FIGURES_DIR = pathlib.Path("figures")
DATA_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

print(f"Ready. This lab needs no data files; save any figures to {FIGURES_DIR.resolve()}")

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.preprocessing import OneHotEncoder

## 1. Univariate model — load data


In [ ]:
people = np.array([1, 2, 3, 4, 5, 1, 2, 3, 4, 5, 1, 2, 3, 4, 5])
trips = np.array([0.9, 1.8, 2.4, 4.3, 6.6, 1.9, 2.7, 3.6, 5.4, 7.7, 2.1, 3.1, 3.7, 5.9, 8.3])
x = people.reshape(-1, 1)
y = trips.reshape(-1, 1)
print(x.shape, y.shape)
print('----------')
trip_df = pd.DataFrame({"people": people, "trips": trips})
trip_df

## 2. Worked example — train/test split and fit

`test_size=0.3` holds out 30% of the rows for testing: the 70% / 30% split from lecture. `random_state=45` makes the split repeatable.

`lm.fit` finds β₀ and β₁ by least squares (the smallest sum of squared errors, SSE). After fitting, `lm.intercept_` is β₀ and `lm.coef_` is β₁ in h(x) = β₀ + β₁x.


In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.3, random_state=45
)
print(x_train.shape, x_test.shape)


In [ ]:
lm = LinearRegression()
lm.fit(x_train, y_train)

y_train_predicted = lm.predict(x_train)
r2_train = r2_score(y_train, y_train_predicted)
n_train, p = x_train.shape
r2_train_adj = 1 - (1 - r2_train) * (n_train - 1) / (n_train - p - 1)

y_test_predicted = lm.predict(x_test)
r2_test = r2_score(y_test, y_test_predicted)
n_test = x_test.shape[0]
r2_test_adj = 1 - (1 - r2_test) * (n_test - 1) / (n_test - p - 1)

print('intercept', lm.intercept_, 'coef', lm.coef_)
print('train R2', r2_train, 'adj', r2_train_adj)
print('test  R2', r2_test, 'adj', r2_test_adj)


Predicted vs measured (1:1 line for comparison), then the residuals. A residual is the error from lecture, e = y − ŷ: measured minus predicted. Plot residuals against the predicted value; a good model leaves them scattered around 0 with no pattern. (Plotted against the measured value, they tilt upward even for a good model.)


In [ ]:
fig, ax = plt.subplots()
ax.plot(y_train, y_train_predicted, marker='o', ls='', label='Training')
ax.plot(y_test, y_test_predicted, marker='o', ls='', label='Testing')
ax.plot([0, 8], [0, 8], ls='--', alpha=0.35, label='1:1')
ax.set_aspect('equal')
ax.set_xlabel('Y measured')
ax.set_ylabel('Y predicted')
ax.legend()
fig.savefig(FIGURES_DIR / 'lab05_univariate_pred.png', dpi=200)


In [ ]:
residual_train = y_train - y_train_predicted
residual_test = y_test - y_test_predicted
fig, ax = plt.subplots()
ax.plot(y_train_predicted, residual_train, marker='o', ls='', label='Training')
ax.plot(y_test_predicted, residual_test, marker='o', ls='', label='Testing')
ax.plot([0, 8], [0, 0], ls='--', alpha=0.35)
ax.set_xlabel('Predicted value')
ax.set_ylabel('Residual')
ax.legend(loc=4)
fig.savefig(FIGURES_DIR / 'lab05_univariate_resid.png', dpi=200)


## 3. Multivariate model with a categorical variable

New today: so far the model has one x (univariate, as in lecture). This one has three x columns: people plus two dummy columns for work status.

`nwork` records how people work: 1 = none, 2 = part-time, 3 = full-time. Lecture calls this kind of variable categorical. One-hot encoding creates a 0/1 column per category. We drop the last column: the three columns always add up to 1, so any one of them can be predicted from the other two (perfect collinearity), and the model could not tell their effects apart.


In [ ]:
nwork = np.array([1, 2, 3, 1, 2, 3, 1, 2, 3, 1, 2, 3, 1, 2, 3]).reshape(-1, 1)
encoder = OneHotEncoder(sparse_output=False)
cat_array = encoder.fit_transform(nwork)
x_with_cat = np.hstack((x, cat_array[:, :2]))
trip_df_cat = pd.DataFrame(
    {"people": people,
     "trips": trips,
     "nwork": nwork.ravel(),
     "cat_1": cat_array[:,0],
     "cat_2": cat_array[:,1],
     "cat_3": cat_array[:,2],
     })
trip_df_cat

In [ ]:
x_train_cat, x_test_cat, y_train_cat, y_test_cat = train_test_split(
    x_with_cat, y, test_size=0.3, random_state=45
)
lm_cat = LinearRegression()
lm_cat.fit(x_train_cat, y_train_cat)

y_train_predicted_cat = lm_cat.predict(x_train_cat)
r2_train_cat = r2_score(y_train_cat, y_train_predicted_cat)
n_train, p_cat = x_train_cat.shape
r2_train_adj_cat = 1 - (1 - r2_train_cat) * (n_train - 1) / (n_train - p_cat - 1)

y_test_predicted_cat = lm_cat.predict(x_test_cat)
r2_test_cat = r2_score(y_test_cat, y_test_predicted_cat)
n_test = x_test_cat.shape[0]
r2_test_adj_cat = 1 - (1 - r2_test_cat) * (n_test - 1) / (n_test - p_cat - 1)

print('intercept', lm_cat.intercept_, 'coef', lm_cat.coef_)
print('train R2', r2_train_cat, 'adj', r2_train_adj_cat)
print('test  R2', r2_test_cat, 'adj', r2_test_adj_cat)


In [ ]:
print('The rsquared value of the training data with 1 covariate was', r2_train)
print('The rsquared value of the training data with 3 covariates was', r2_train_cat)
print('The adjusted rsquared value of the training data with 1 covariate was', r2_train_adj)
print('The adjusted rsquared value of the training data with 3 covariates was', r2_train_adj_cat)
print('--------------------------')
print('The rsquared value of the testing data with 1 covariate was', r2_test)
print('The rsquared value of the testing data with 3 covariates was', r2_test_cat)
print('The adjusted rsquared value of the testing data with 1 covariate was', r2_test_adj)
print('The adjusted rsquared value of the testing data with 3 covariates was', r2_test_adj_cat)


## 4. Wrap-up

Training R² often rises when you add variables; test R² (and adjusted R²) tell you whether that extra complexity actually generalizes. Continue with `lab05-activity.ipynb`.
